In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

from sklearn import preprocessing
from sklearn.model_selection import KFold

from ipywidgets import interact, widgetss

# Remove all the warnings
import warnings
warnings.filterwarnings('ignore')


device = torch.device("cpu")

# Retina display
%config InlineBackend.figure_format = 'retina'

from einops import rearrange
from tqdm import tqdm,trange

In [ ]:
img = torchvision.io.read_image("dog.jpg")
print(img.shape)

In [ ]:
plt.imshow(rearrange(img, 'c h w -> h w c').numpy()) # Display the full image

In [ ]:
from sklearn import preprocessing

scaler_img = preprocessing.MinMaxScaler().fit(img.reshape(-1, 1))
scaler_img

In [ ]:
img_scaled = scaler_img.transform(img.reshape(-1, 1)).reshape(img.shape)
img_scaled.shape

img_scaled = torch.tensor(img_scaled)

In [ ]:
img_scaled

In [ ]:
crop = torchvision.transforms.functional.crop(img_scaled, 600, 800, 300, 300)
crop.shape

In [ ]:
plt.imshow(rearrange(crop, 'c h w -> h w c').cpu().numpy())

In [ ]:
mseloss_dict = {}
psnr_dict = {}

In [ ]:
def factorize(A, k):
    W = torch.randn(A.shape[0], k, requires_grad=True, device=device)
    H = torch.randn(k, A.shape[1], requires_grad=True, device=device)
    optimizer = optim.Adam([W, H], lr=0.01)
    mask = ~torch.isnan(A)
    tol = 1e-5
    for i in range(1000):
        diff_matrix = torch.mm(W, H) - A
        diff_vector = diff_matrix[mask]
        loss = torch.norm(diff_vector)
        if loss < tol:
            break
        
        optimizer.zero_grad()
        
        loss.backward()
        
        optimizer.step()
        
    return W, H, loss

In [ ]:
def nested_cross_validation(k_max,outer_folds,inner_folds, A):
    hyperparameters = {k for k in range(2,k_max+1,10)}
    num_outer_folds = outer_folds
    num_inner_folds = inner_folds

    kf_outer = KFold(n_splits=num_outer_folds, shuffle=False)
    kf_inner = KFold(n_splits=num_inner_folds, shuffle=False)

    results= {}
    outer_count = 0
    overall_count = 0
    for outer_train_index, outer_test_index in kf_outer.split(A):
        X_outer_train, X_outer_test = A[outer_train_index], A[outer_test_index] 

        inner_count = 0

        for innner_train_index, inner_test_index in kf_inner.split(X_outer_train):
            print("Outer Fold {}, Inner Fold {}".format(outer_count+1, inner_count+1))
            X_inner_train, X_inner_test = X_outer_train[innner_train_index], X_outer_train[inner_test_index]

            for k in hyperparameters:
                W,H,loss = factorize(X_inner_train, k)

                results[overall_count] = {'outer_fold': outer_count, 'inner_fold': inner_count, '# of factors': k, 'loss': loss}
                overall_count += 1

            inner_count += 1
        outer_count += 1
    overall_results = pd.DataFrame(results).T
    ans = np.array([])
    for fold in range(num_outer_folds):
        outer_fold_df = overall_results.query('outer_fold == @fold')
        sorted_df = outer_fold_df.sort_values(by='loss', ascending=True)
        best_k = sorted_df[['# of factors', 'loss']].head(1)['# of factors'].values[0]
        ans = np.append(ans, best_k)
    print(f'Best k value(aggregate) = {int(np.mean(ans))}')
    return 0

In [ ]:
import torch
import torch.nn.functional as F

def calculate_rmse(image1, image2):
    mse = F.mse_loss(image1, image2)
    rmse = torch.sqrt(mse)
    print(f"RMSE between the two images: {rmse.item()}")
    return rmse.item()

In [ ]:
import torch
import torch.nn.functional as F

def calculate_psnr(original, reconstructed, max_pixel_value=1.0):

    mse = F.mse_loss(original, reconstructed)
    psnr = 10 * torch.log10((max_pixel_value ** 2) / mse)
    print(f"PSNR between the two images: {psnr.item()} dB")
    return psnr.item()

In [ ]:
# N = 30

img_30 = crop.clone()
img_30[:, 50:80, 150:180] = float('nan')
W_30_r,H_30_r,loss_30_r = factorize(img_30[0,:,:], 92)
W_30_g,H_30_g,loss_30_g = factorize(img_30[1,:,:], 92)
W_30_b,H_30_b,loss_30_b = factorize(img_30[2,:,:], 92)
reconstructed_30_r = torch.mm(W_30_r, H_30_r)
reconstructed_30_g = torch.mm(W_30_g, H_30_g)
reconstructed_30_b = torch.mm(W_30_b, H_30_b)
reconstructed_30 = torch.cat((reconstructed_30_r.unsqueeze(0), reconstructed_30_g.unsqueeze(0), reconstructed_30_b.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(img_30, 'c h w -> h w c').numpy())
axes[0].set_title('Image with missing patch')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

reconstructed_30[:, 50:80, 150:180] = (reconstructed_30[:, 50:80, 150:180] - reconstructed_30[:, 50:80, 150:180].min()) / (reconstructed_30[:, 50:80, 150:180].max() - reconstructed_30[:, 50:80, 150:180].min())
axes[2].imshow(rearrange(reconstructed_30.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

mseloss_dict[30] = calculate_rmse(reconstructed_30, crop)
psnr_dict[30] = calculate_psnr(crop, reconstructed_30)

In [ ]:
# N = 20 

img_20 = crop.clone()  # Replace img_30 with img_20
img_20[:, 50:70, 150:170] = float('nan')
W_20_r, H_20_r, loss_20_r = factorize(img_20[0,:,:], 92)
W_20_g, H_20_g, loss_20_g = factorize(img_20[1,:,:], 92)
W_20_b, H_20_b, loss_20_b = factorize(img_20[2,:,:], 92)
reconstructed_20_r = torch.mm(W_20_r, H_20_r)
reconstructed_20_g = torch.mm(W_20_g, H_20_g)
reconstructed_20_b = torch.mm(W_20_b, H_20_b)
reconstructed_20 = torch.cat((reconstructed_20_r.unsqueeze(0), reconstructed_20_g.unsqueeze(0), reconstructed_20_b.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(img_20, 'c h w -> h w c').numpy())
axes[0].set_title('Image with missing patch')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

reconstructed_20[:, 50:70, 150:170] = (reconstructed_20[:, 50:70, 150:170] - reconstructed_20[:, 50:70, 150:170].min()) / (reconstructed_20[:, 50:70, 150:170].max() - reconstructed_20[:, 50:70, 150:170].min())
axes[2].imshow(rearrange(reconstructed_20.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')
                  
mseloss_dict[20] = calculate_rmse(reconstructed_20, crop)
psnr_dict[20] = calculate_psnr(crop, reconstructed_20)

In [ ]:
# N = 40 

img_40 = crop.clone()  # Replace img_30 with img_40
img_40[:, 50:90, 150:190] = float('nan')
W_40_r, H_40_r, loss_40_r = factorize(img_40[0,:,:], 92)
W_40_g, H_40_g, loss_40_g = factorize(img_40[1,:,:], 92)
W_40_b, H_40_b, loss_40_b = factorize(img_40[2,:,:], 92)
reconstructed_40_r = torch.mm(W_40_r, H_40_r)
reconstructed_40_g = torch.mm(W_40_g, H_40_g)
reconstructed_40_b = torch.mm(W_40_b, H_40_b)
reconstructed_40 = torch.cat((reconstructed_40_r.unsqueeze(0), reconstructed_40_g.unsqueeze(0), reconstructed_40_b.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(img_40, 'c h w -> h w c').numpy())
axes[0].set_title('Image with missing patch')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

reconstructed_40[:, 50:90, 150:190] = (reconstructed_40[:,50:90,150:190] - reconstructed_40[:,50:90,150:190].min()) / (reconstructed_40[:,50:90,150:190].max() - reconstructed_40[:,50:90,150:190].min())
axes[2].imshow(rearrange(reconstructed_40.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

mseloss_dict[40] = calculate_rmse(reconstructed_40, crop)
psnr_dict[40] = calculate_psnr(crop, reconstructed_40)

In [ ]:
# N = 60 

img_60 = crop.clone()  
img_60[:, 50:110, 150:210] = float('nan')
W_60_r, H_60_r, loss_60_r = factorize(img_60[0,:,:], 92)
W_60_g, H_60_g, loss_60_g = factorize(img_60[1,:,:], 92)
W_60_b, H_60_b, loss_60_b = factorize(img_60[2,:,:], 92)
reconstructed_60_r = torch.mm(W_60_r, H_60_r)
reconstructed_60_g = torch.mm(W_60_g, H_60_g)
reconstructed_60_b = torch.mm(W_60_b, H_60_b)
reconstructed_60 = torch.cat((reconstructed_60_r.unsqueeze(0), reconstructed_60_g.unsqueeze(0), reconstructed_60_b.unsqueeze(0)), dim=0)
           
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(img_60, 'c h w -> h w c').numpy())
axes[0].set_title('Image with missing patch')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

reconstructed_60[:, 50:110, 150:210] = (reconstructed_60[:,50:110,150:210] - reconstructed_60[:,50:110,150:210].min()) / (reconstructed_60[:,50:110,150:210].max() - reconstructed_60[:,50:110,150:210].min())
axes[2].imshow(rearrange(reconstructed_60.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

mseloss_dict[60] = calculate_rmse(reconstructed_60, crop)
psnr_dict[60] = calculate_psnr(crop, reconstructed_60)

In [ ]:
img_80 = crop.clone() 
img_80[:, 50:130, 150:230] = float('nan')
W_80_r, H_80_r, loss_80_r = factorize(img_80[0,:,:], 92)
W_80_g, H_80_g, loss_80_g = factorize(img_80[1,:,:], 92)
W_80_b, H_80_b, loss_80_b = factorize(img_80[2,:,:], 92)
reconstructed_80_r = torch.mm(W_80_r, H_80_r)
reconstructed_80_g = torch.mm(W_80_g, H_80_g)
reconstructed_80_b = torch.mm(W_80_b, H_80_b)
reconstructed_80 = torch.cat((reconstructed_80_r.unsqueeze(0), reconstructed_80_g.unsqueeze(0), reconstructed_80_b.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(img_80, 'c h w -> h w c').numpy())
axes[0].set_title('Image with missing patch')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

reconstructed_80[:,50:130,150:230] = (reconstructed_80[:,50:130,150:230] - reconstructed_80[:,50:130,150:230].min()) / (reconstructed_80[:,50:130,150:230].max() - reconstructed_80[:,50:130,150:230].min())
axes[2].imshow(rearrange(reconstructed_80.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

mseloss_dict[80] = calculate_rmse(reconstructed_80, crop)
psnr_dict[80] = calculate_psnr(crop, reconstructed_80)

In [ ]:
random_rmse_dict = {}
random_psnr_dict = {}

In [ ]:
# N = 30

random_30 = crop.clone()
n = 30
pixels_to_remove = n**2
indices_to_remove = np.random.choice(300*300, pixels_to_remove, replace=False)
row_indices, col_indices = np.unravel_index(indices_to_remove, random_30[0,:,:].shape)
random_30[0,:,:][row_indices, col_indices] = float('nan')
random_30[1,:,:][row_indices, col_indices] = float('nan')
random_30[2,:,:][row_indices, col_indices] = float('nan')

W_r_30, H_r_30, loss_r_30 = factorize(random_30[0,:,:], 92)
W_g_30, H_g_30, loss_g_30 = factorize(random_30[1,:,:], 92)
W_b_30, H_b_30, loss_b_30 = factorize(random_30[2,:,:], 92)
reconstructed_r_30 = torch.mm(W_r_30, H_r_30)
reconstructed_g_30 = torch.mm(W_g_30, H_g_30)
reconstructed_b_30 = torch.mm(W_b_30, H_b_30)
reconstructed_30_random = torch.cat((reconstructed_r_30.unsqueeze(0), reconstructed_g_30.unsqueeze(0), reconstructed_b_30.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(random_30, 'c h w -> h w c').numpy())
axes[0].set_title('Image with random missing pixels')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

axes[2].imshow(rearrange(reconstructed_30_random.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

random_rmse_dict[30] = calculate_rmse(reconstructed_30_random, crop)
random_psnr_dict[30] = calculate_psnr(crop, reconstructed_30_random)

In [ ]:
# N = 20

random_20 = crop.clone()
n = 20
pixels_to_remove = n**2
indices_to_remove = np.random.choice(300*300, pixels_to_remove, replace=False)
row_indices, col_indices = np.unravel_index(indices_to_remove, random_20[0,:,:].shape)
random_20[0,:,:][row_indices, col_indices] = float('nan')
random_20[1,:,:][row_indices, col_indices] = float('nan')
random_20[2,:,:][row_indices, col_indices] = float('nan')

W_r_20, H_r_20, loss_r_20 = factorize(random_20[0,:,:], 92)
W_g_20, H_g_20, loss_g_20 = factorize(random_20[1,:,:], 92)
W_b_20, H_b_20, loss_b_20 = factorize(random_20[2,:,:], 92)
reconstructed_r_20 = torch.mm(W_r_20, H_r_20)
reconstructed_g_20 = torch.mm(W_g_20, H_g_20)
reconstructed_b_20 = torch.mm(W_b_20, H_b_20)
reconstructed_20_random = torch.cat((reconstructed_r_20.unsqueeze(0), reconstructed_g_20.unsqueeze(0), reconstructed_b_20.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(random_20, 'c h w -> h w c').numpy())
axes[0].set_title('Image with random missing pixels')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

axes[2].imshow(rearrange(reconstructed_20_random.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

random_rmse_dict[20] = calculate_rmse(reconstructed_20_random, crop)
random_psnr_dict[20] = calculate_psnr(crop, reconstructed_20_random)

In [ ]:
# N = 40

random_40 = crop.clone()
n = 40
pixels_to_remove = n**2
indices_to_remove = np.random.choice(300*300, pixels_to_remove, replace=False)
row_indices, col_indices = np.unravel_index(indices_to_remove, random_40[0,:,:].shape)
random_40[0,:,:][row_indices, col_indices] = float('nan')
random_40[1,:,:][row_indices, col_indices] = float('nan')
random_40[2,:,:][row_indices, col_indices] = float('nan')

W_r_40, H_r_40, loss_r_40 = factorize(random_40[0,:,:], 92)
W_g_40, H_g_40, loss_g_40 = factorize(random_40[1,:,:], 92)
W_b_40, H_b_40, loss_b_40 = factorize(random_40[2,:,:], 92)
reconstructed_r_40 = torch.mm(W_r_40, H_r_40)
reconstructed_g_40 = torch.mm(W_g_40, H_g_40)
reconstructed_b_40 = torch.mm(W_b_40, H_b_40)
reconstructed_40_random = torch.cat((reconstructed_r_40.unsqueeze(0), reconstructed_g_40.unsqueeze(0), reconstructed_b_40.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(random_40, 'c h w -> h w c').numpy())
axes[0].set_title('Image with random missing pixels')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

axes[2].imshow(rearrange(reconstructed_40_random.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

random_rmse_dict[40] = calculate_rmse(reconstructed_40_random, crop)
random_psnr_dict[40] = calculate_psnr(crop, reconstructed_40_random)

In [ ]:
# N = 60

random_60 = crop.clone()
n = 60
pixels_to_remove = n**2
indices_to_remove = np.random.choice(300*300, pixels_to_remove, replace=False)
row_indices, col_indices = np.unravel_index(indices_to_remove, random_60[0,:,:].shape)
random_60[0,:,:][row_indices, col_indices] = float('nan')
random_60[1,:,:][row_indices, col_indices] = float('nan')
random_60[2,:,:][row_indices, col_indices] = float('nan')

W_r_60, H_r_60, loss_r_60 = factorize(random_60[0,:,:], 92)
W_g_60, H_g_60, loss_g_60 = factorize(random_60[1,:,:], 92)
W_b_60, H_b_60, loss_b_60 = factorize(random_60[2,:,:], 92)
reconstructed_r_60 = torch.mm(W_r_60, H_r_60)
reconstructed_g_60 = torch.mm(W_g_60, H_g_60)
reconstructed_b_60 = torch.mm(W_b_60, H_b_60)
reconstructed_60_random = torch.cat((reconstructed_r_60.unsqueeze(0), reconstructed_g_60.unsqueeze(0), reconstructed_b_60.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(random_60, 'c h w -> h w c').numpy())
axes[0].set_title('Image with random missing pixels')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

axes[2].imshow(rearrange(reconstructed_60_random.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

random_rmse_dict[60] = calculate_rmse(reconstructed_60_random, crop)
random_psnr_dict[60] = calculate_psnr(crop, reconstructed_60_random)

In [ ]:
# N = 80

random_80 = crop.clone()
n = 80
pixels_to_remove = n**2
indices_to_remove = np.random.choice(300*300, pixels_to_remove, replace=False)
row_indices, col_indices = np.unravel_index(indices_to_remove, random_80[0,:,:].shape)
random_80[0,:,:][row_indices, col_indices] = float('nan')
random_80[1,:,:][row_indices, col_indices] = float('nan')
random_80[2,:,:][row_indices, col_indices] = float('nan')

W_r_80, H_r_80, loss_r_80 = factorize(random_80[0,:,:], 92)
W_g_80, H_g_80, loss_g_80 = factorize(random_80[1,:,:], 92)
W_b_80, H_b_80, loss_b_80 = factorize(random_80[2,:,:], 92)
reconstructed_r_80 = torch.mm(W_r_80, H_r_80)
reconstructed_g_80 = torch.mm(W_g_80, H_g_80)
reconstructed_b_80 = torch.mm(W_b_80, H_b_80)
reconstructed_80_random = torch.cat((reconstructed_r_80.unsqueeze(0), reconstructed_g_80.unsqueeze(0), reconstructed_b_80.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(random_80, 'c h w -> h w c').numpy())
axes[0].set_title('Image with random missing pixels')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

axes[2].imshow(rearrange(reconstructed_80_random.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

random_rmse_dict[80] = calculate_rmse(reconstructed_80_random, crop)
random_psnr_dict[80] = calculate_psnr(crop, reconstructed_80_random)

In [ ]:
def als(matrix, rank, max_iter=1000, tol=1e-4):
    """
        matrix (torch.Tensor): The input matrix.
        rank (int): The desired rank of the factorization.
        max_iter (int): Maximum number of iterations.
        tol (float): Convergence tolerance
        
        W (torch.Tensor): Factor matrix W.
        H (torch.Tensor): Factor matrix H.
        error (float): Reconstruction error.
    """
    matrix = matrix.float()  # Ensure input matrix is of type float32
    W = torch.rand(matrix.shape[0], rank)
    H = torch.rand(rank, matrix.shape[1])

    for _ in trange(max_iter):
        # Update W
        for i in range(matrix.shape[0]):
            mask = ~torch.isnan(matrix[i, :])
            H_masked = H[:, mask]
            A_masked = matrix[i, mask].T

            # Cast solution
            s = torch.linalg.lstsq( H_masked.T, A_masked).solution
            W[i,:]=s.T

        # Update H
        for j in range(matrix.shape[1]):
            mask = ~torch.isnan(matrix[:, j])
            W_masked = W[mask, :]
            A_masked = matrix[mask, j]

            # Cast solution to float
            H[:, j] = torch.linalg.lstsq(W_masked, A_masked).solution

        # Check for convergence
        reconstruction = W @ H
        error = torch.sqrt(torch.mean((matrix - reconstruction)**2))
        if error < tol:
            break

    return W, H, error

In [ ]:
plt.figure(figsize=(10, 5))
xs, ys = zip(*sorted(zip(mseloss_dict.keys(), mseloss_dict.values())))
plt.plot(xs, ys, label='RMSE')
plt.xlabel('Patch Size')
plt.ylabel('RMSE')
plt.title('RMSE vs Patch Size')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
xs, ys = zip(*sorted(zip(psnr_dict.keys(), psnr_dict.values())))
plt.plot(xs, ys, label='PSNR')
plt.xlabel('Patch Size')
plt.ylabel('PSNR')
plt.title('PSNR vs Patch Size')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
xs, ys = zip(*sorted(zip(random_rmse_dict.keys(), random_rmse_dict.values())))
plt.plot(xs, ys, label='RMSE')
plt.xlabel('Patch Size of Random Points')
plt.ylabel('RMSE')
plt.title('RMSE vs Patch Size of Random Points')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
xs, ys = zip(*sorted(zip(random_psnr_dict.keys(), random_psnr_dict.values())))
plt.plot(xs, ys, label='PSNR')
plt.xlabel('Patch Size of Random Points')
plt.ylabel('PSNR')
plt.title('PSNR vs Patch Size of Random Points')
plt.show()


ALS'

In [ ]:
# N = 20

random_20 = crop.clone()
n = 20
pixels_to_remove = n**2
indices_to_remove = np.random.choice(300*300, pixels_to_remove, replace=False)
row_indices, col_indices = np.unravel_index(indices_to_remove, random_20[0,:,:].shape)
random_20[0,:,:][row_indices, col_indices] = float('nan')
random_20[1,:,:][row_indices, col_indices] = float('nan')
random_20[2,:,:][row_indices, col_indices] = float('nan')

W_r_20, H_r_20, loss_r_20 = als(random_20[0,:,:], 92)
W_g_20, H_g_20, loss_g_20 = als(random_20[1,:,:], 92)
W_b_20, H_b_20, loss_b_20 = als(random_20[2,:,:], 92)
reconstructed_r_20 = torch.mm(W_r_20, H_r_20)
reconstructed_g_20 = torch.mm(W_g_20, H_g_20)
reconstructed_b_20 = torch.mm(W_b_20, H_b_20)
reconstructed_20_random = torch.cat((reconstructed_r_20.unsqueeze(0), reconstructed_g_20.unsqueeze(0), reconstructed_b_20.unsqueeze(0)), dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rearrange(random_20, 'c h w -> h w c').numpy())
axes[0].set_title('Image with random missing pixels')

axes[1].imshow(rearrange(crop, 'c h w -> h w c').numpy())
axes[1].set_title('Original Image')

axes[2].imshow(rearrange(reconstructed_20_random.detach().numpy(), 'c h w -> h w c'))
axes[2].set_title('Reconstructed Image')

als_dict_rmse['20'] = calculate_rmse(reconstructed_20_random, crop)
als_dict_psnr['20'] = calculate_psnr(crop, reconstructed_20_random)